# 🎯 Verificación multi-semilla del AUC — Modelo binario de zona (Mac / MPS)

Reentrena tu **modelo binario de zona** (el del AUC 0,93, tu resultado estrella) con **varias semillas** para reportar el AUC como **media ± desviación**, y confirmar que el número se sostiene antes de enviarlo al congreso.

**Por qué importa:** acabamos de ver que el F1 del auditor variaba ±0,05 entre semillas. El AUC de 0,93 nunca se promedió: pudo ser una corrida afortunada. Esta verificación entrega el número honesto y defendible.

**Diseño:** test fijo (semilla 42, mismos 11 casos de riesgo). Varía solo la estocasticidad del entrenamiento. Tarea binaria: zona segura (BI-RADS 0–3) = 0 · zona de riesgo (4–6) = 1.

⚠️ **Tiempo:** 3 entrenamientos (~45 min–1,5 h en MPS). Baja `SEEDS` a 2 si necesitas.

---
**Uso:** en `notebooks/`. Lee `../data/processed/dataset_clean.csv`. Restart + Run All.

## 1 · Configuración

In [ ]:
import numpy as np, pandas as pd, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, confusion_matrix

DEVICE='cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
MAX_LENGTH=256; LR=3e-5; EPOCHS=15; PATIENCE=4; BATCH=16
SEEDS=[42,123,7]          # baja a [42,123] si quieres menos tiempo
SPLIT_SEED=42
print("Dispositivo:", DEVICE, "| semillas:", SEEDS)

## 2 · Datos y split FIJO (zona binaria)

In [ ]:
df=pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X=df['auditor_input'].values; y7=df['BI-RADS'].values
# split estratificado por las 7 clases (mismo test que el resto de experimentos)
Xtv,Xte,y7tv,y7te=train_test_split(X,y7,test_size=0.15,random_state=SPLIT_SEED,stratify=y7)
Xtr,Xva,y7tr,y7va=train_test_split(Xtv,y7tv,test_size=0.176,random_state=SPLIT_SEED,stratify=y7tv)
# a binario zona
z=lambda v:(np.asarray(v)>=4).astype(int)
ytr,yva,yte = z(y7tr), z(y7va), z(y7te)
print(f"Train {len(Xtr)} (riesgo {ytr.sum()}) | Val {len(Xva)} (riesgo {yva.sum()}) | Test {len(Xte)} (riesgo {yte.sum()})")

## 3 · Dataset, modelo binario y augmentación

In [ ]:
try: TOK=AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; TOK=AutoTokenizer.from_pretrained(MODELO)
class DS(Dataset):
    def __init__(self,t,l): self.t=list(t); self.l=list(l)
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=TOK(str(self.t[i]),truncation=True,padding='max_length',max_length=MAX_LENGTH,
              return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}
class ZonaRoBERTa(nn.Module):
    def __init__(self,m,dp=0.3):
        super().__init__(); self.enc=AutoModel.from_pretrained(m)
        self.dp=nn.Dropout(dp); self.cl=nn.Linear(self.enc.config.hidden_size,2)
    def forward(self,ids,mask):
        return self.cl(self.dp(self.enc(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]))
def aumentar(t):
    V=[('bordes irregulares','márgenes irregulares'),('bordes espiculados','márgenes espiculados'),
       ('bordes mal definidos','límites imprecisos'),('imagen nodular','nódulo'),('nódulo','imagen nodular'),
       ('lesión nodular','nódulo'),('heterogéneamente densas','de densidad heterogénea'),
       ('muy densas','extremadamente densas'),('calcificaciones sospechosas','depósitos cálcicos sospechosos'),
       ('microcalcificaciones','calcificaciones puntiformes agrupadas'),('mama derecha','hemimama derecha'),
       ('mama izquierda','hemimama izquierda'),('se visualiza','se observa'),('se observa','se visualiza'),
       ('se evidencia','se observa')]
    s=t
    for o,r in random.sample(V,min(3,len(V))):
        if o in s: s=s.replace(o,r,1)
    return s
print("Listo.")

## 4 · Entrenamiento binario (re-sembrado por corrida)

In [ ]:
def entrenar_zona(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE=='mps': torch.mps.manual_seed(seed)
    # Augmentación de la zona de riesgo (minoritaria) - solo train
    mask=(ytr==1); ta,la=[],[]
    for txt in Xtr[mask]:
        for _ in range(3): ta.append(aumentar(txt)); la.append(1)
    Xa=np.concatenate([Xtr,np.array(ta)]); ya=np.concatenate([ytr,np.array(la)])
    idx=np.random.RandomState(seed).permutation(len(Xa)); Xa,ya=Xa[idx],ya[idx]

    tl=DataLoader(DS(Xa,ya),batch_size=BATCH,shuffle=True)
    vl=DataLoader(DS(Xva,yva),batch_size=BATCH); tt=DataLoader(DS(Xte,yte),batch_size=BATCH)
    model=ZonaRoBERTa(MODELO).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    sch=get_linear_schedule_with_warmup(opt,0,len(tl)*EPOCHS)
    pw=compute_class_weight('balanced',classes=np.array([0,1]),y=ya)
    crit=nn.CrossEntropyLoss(weight=torch.tensor(pw,dtype=torch.float32).to(DEVICE))

    def ep(loader,train=True):
        model.train() if train else model.eval(); P=[]; R=[]
        with torch.set_grad_enabled(train):
            for b in loader:
                ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE); lb=b['labels'].to(DEVICE)
                if train: opt.zero_grad()
                lo=model(ids,mk); loss=crit(lo,lb)
                if train: loss.backward(); opt.step(); sch.step()
                P.extend(torch.softmax(lo,1)[:,1].detach().cpu().numpy()); R.extend(lb.cpu().numpy())
        try: return roc_auc_score(R,P)
        except: return 0.5
    mejor,sin=0,0; ruta=f'tmp_zona_{seed}.pt'
    for e in range(1,EPOCHS+1):
        _=ep(tl,True); va=ep(vl,False)
        if va>mejor: mejor,sin=va,0; torch.save(model.state_dict(),ruta)
        else: sin+=1
        if sin>=PATIENCE: break
    model.load_state_dict(torch.load(ruta,map_location=DEVICE)); model.eval()
    P=[]; R=[]
    with torch.no_grad():
        for b in tt:
            ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE)
            P.extend(torch.softmax(model(ids,mk).float(),1)[:,1].cpu().numpy()); R.extend(b['labels'].numpy())
    P=np.array(P); R=np.array(R); auc=roc_auc_score(R,P)
    yp=(P>=0.5).astype(int); tn,fp,fn,tp=confusion_matrix(R,yp,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
    return {'auc':auc,'sens':sens,'espec':espec}

## 5 · Ejecutar las corridas

In [ ]:
res=[]
for s in SEEDS:
    print(f"▶ semilla {s} ...", flush=True)
    r=entrenar_zona(s); res.append(r)
    print(f"   AUC={r['auc']:.4f} | Sens={r['sens']:.3f} | Espec={r['espec']:.3f}", flush=True)
print("\nListo.")

## 6 · Resumen: AUC media ± desviación

In [ ]:
aucs=[r['auc'] for r in res]; sens=[r['sens'] for r in res]; esp=[r['espec'] for r in res]
print("="*56)
print(f"MODELO BINARIO DE ZONA — multi-semilla (n={len(SEEDS)}, test fijo)")
print("="*56)
print(f"  AUC-ROC:        {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Sensibilidad:   {np.mean(sens):.3f} ± {np.std(sens):.3f}")
print(f"  Especificidad:  {np.mean(esp):.3f} ± {np.std(esp):.3f}")
print("="*56)
print("  AUC por semilla:", ", ".join(f"{a:.4f}" for a in aucs))
print(f"\n  Comparar contra el 0,93 de una sola corrida.")
print("  El valor a REPORTAR en el congreso es la media ± desviación.")

## 7 · Cómo interpretar y qué reportar

- El número honesto para el congreso es la **media ± desviación** del AUC, no el 0,93 de una corrida.
- Si la media queda cerca de 0,93 con desviación baja (p. ej. 0,92 ± 0,01), tu cifra estrella se sostiene: excelente, repórtala con el intervalo.
- Si la media baja (p. ej. 0,88 ± 0,03), ese es el valor correcto y ajustamos el resumen del congreso.
- En cualquier caso, reportar media ± desviación es más sólido y defendible que un número único.

Texto sugerido: *"El modelo discrimina zona segura de riesgo con un AUC de X ± Y (promedio de N semillas)."*